## 18. Hybrid Mamba+Attn
**Paper / idea:** Jamba 2024, Samba 2024

Mamba layers with a full-attention layer every 4th layer, plus MLPs. Tests whether a little attention fixes recall weaknesses of pure SSMs.

**Setup:** TinyStories, 4096-token BPE, context 256, ~10M parameters, 4000 steps x 32 x 256 tokens, fp16 on a T4.
Run top to bottom (Runtime -> Run all). On Kaggle turn Internet **On**. Outputs: `results/hybrid.json`, `ckpt/hybrid.pt`, and `hybrid_10m.zip`.

In [ ]:
!pip -q install datasets tokenizers matplotlib

In [ ]:
NAME = "hybrid"
STEPS = 4000   # lower (e.g. 2000) for a faster run
V, SEQ, EOS = 4096, 256, 0
import os, math, json, time
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

In [ ]:
# ---- data: TinyStories + BPE(4096) -> data/*.npy
def prep_data(out="data", n_train=300_000, n_val=5_000):
    if os.path.exists(f"{out}/train.npy"): return
    os.makedirs(out, exist_ok=True)
    from datasets import load_dataset
    from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
    ds = load_dataset("roneneldan/TinyStories")
    tr = ds["train"].select(range(n_train))["text"]; va = ds["validation"].select(range(n_val))["text"]
    tok = Tokenizer(models.BPE()); tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tok.decoder = decoders.ByteLevel()
    tok.train_from_iterator(tr[:100_000], trainers.BpeTrainer(
        vocab_size=V, special_tokens=["<eos>"], initial_alphabet=pre_tokenizers.ByteLevel.alphabet()))
    tok.save(f"{out}/tok.json")
    def enc(texts):
        parts = []
        for i in range(0, len(texts), 20_000):
            ids = []
            for e in tok.encode_batch(texts[i:i + 20_000]): ids.extend(e.ids); ids.append(EOS)
            parts.append(np.array(ids, dtype=np.uint16))
        return np.concatenate(parts)
    for n, t in [("train", tr), ("val", va)]:
        a = enc(t); np.save(f"{out}/{n}.npy", a); print(n, "tokens:", len(a))

def get_batch(data, bs, T, g=None):
    ix = torch.randint(0, len(data) - T - 1, (bs,), generator=g).to(data.device)
    idx = ix[:, None] + torch.arange(T + 1, device=data.device)
    b = data[idx].long(); return b[:, :-1], b[:, 1:]

prep_data()

In [ ]:
# ---- shared building blocks
def nocast(): return torch.autocast("cuda", enabled=False)

def _init(m):
    if isinstance(m, (nn.Linear, nn.Embedding)): nn.init.normal_(m.weight, std=0.02)

class RMSNorm(nn.Module):
    def __init__(s, d, eps=1e-6): super().__init__(); s.w = nn.Parameter(torch.ones(d)); s.eps = eps
    def forward(s, x):
        xf = x.float(); return (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + s.eps)).type_as(x) * s.w

class Block(nn.Module):
    def __init__(s, d, mixer, ffn=None, norm=RMSNorm):
        super().__init__(); s.n1 = norm(d); s.mixer = mixer
        s.n2 = norm(d) if ffn is not None else None; s.ffn = ffn
    def forward(s, x):
        x = x + s.mixer(s.n1(x))
        return x + s.ffn(s.n2(x)) if s.ffn is not None else x

class LM(nn.Module):
    def __init__(s, d, blocks, pos=False, ckpt=False, norm=RMSNorm):
        super().__init__()
        s.emb = nn.Embedding(V, d); s.pos = nn.Embedding(SEQ, d) if pos else None
        s.blocks = nn.ModuleList(blocks); s.norm = norm(d); s.ckpt = ckpt
        s.head = nn.Linear(d, V, bias=False); s.apply(_init); s.head.weight = s.emb.weight
    def forward(s, idx):
        x = s.emb(idx)
        if s.pos is not None: x = x + s.pos(torch.arange(idx.shape[1], device=idx.device))
        for b in s.blocks:
            x = checkpoint(b, x, use_reentrant=False) if (s.ckpt and s.training) else b(x)
        return s.head(s.norm(x))

In [ ]:
# ---- the model: Hybrid Mamba+Attn
def rope(x, base=10000.0):  # x (B,H,T,hd)
    T, hd = x.shape[2], x.shape[3]
    inv = 1.0 / (base ** (torch.arange(0, hd, 2, device=x.device).float() / hd))
    ang = torch.outer(torch.arange(T, device=x.device).float(), inv)
    cos, sin = ang.cos().to(x.dtype), ang.sin().to(x.dtype)
    x1, x2 = x[..., ::2], x[..., 1::2]
    return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], -1).flatten(-2)

def lin_scan(a, b):
    """h_t = a_t*h_{t-1} + b_t along dim 1 (Hillis-Steele parallel scan, O(T log T), exact)."""
    T, d = a.shape[1], 1
    while d < T:
        b = torch.cat([b[:, :d], b[:, d:] + a[:, d:] * b[:, :-d]], 1)
        a = torch.cat([a[:, :d], a[:, d:] * a[:, :-d]], 1); d *= 2
    return b

def _dt_bias(n):
    dt = torch.exp(torch.rand(n) * (math.log(0.1) - math.log(0.001)) + math.log(0.001)).clamp(min=1e-4)
    return dt + torch.log(-torch.expm1(-dt))

class Attn(nn.Module):
    def __init__(s, d, H, kv=None, rope=False, window=None):
        super().__init__(); s.H, s.kv, s.hd, s.rope, s.window = H, kv or H, d // H, rope, window
        s.qkv = nn.Linear(d, (H + 2 * s.kv) * s.hd, bias=False); s.o = nn.Linear(d, d, bias=False)
    def forward(s, x):
        B, T, _ = x.shape; H, kv, hd = s.H, s.kv, s.hd
        q, k, v = s.qkv(x).split([H * hd, kv * hd, kv * hd], -1)
        q = q.view(B, T, H, hd).transpose(1, 2); k = k.view(B, T, kv, hd).transpose(1, 2); v = v.view(B, T, kv, hd).transpose(1, 2)
        if s.rope: q, k = rope(q), rope(k)
        if kv != H: k, v = k.repeat_interleave(H // kv, 1), v.repeat_interleave(H // kv, 1)
        if s.window:
            i = torch.arange(T, device=x.device)
            m = (i[:, None] >= i[None]) & (i[:, None] - i[None] < s.window)
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=m)
        else:
            y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return s.o(y.transpose(1, 2).reshape(B, T, -1))

class Mamba(nn.Module):  # selective SSM (S6), parallel scan
    def __init__(s, d, E=2, N=16, conv=4):
        super().__init__(); s.Ei, s.N = E * d, N; s.dtr = math.ceil(d / 16)
        s.in_proj = nn.Linear(d, 2 * s.Ei, bias=False); s.conv = nn.Conv1d(s.Ei, s.Ei, conv, groups=s.Ei, padding=conv - 1)
        s.x_proj = nn.Linear(s.Ei, s.dtr + 2 * N, bias=False); s.dt_proj = nn.Linear(s.dtr, s.Ei)
        with torch.no_grad(): s.dt_proj.bias.copy_(_dt_bias(s.Ei))
        s.A_log = nn.Parameter(torch.log(torch.arange(1, N + 1).float()).repeat(s.Ei, 1))
        s.D = nn.Parameter(torch.ones(s.Ei)); s.out_proj = nn.Linear(s.Ei, d, bias=False)
    def forward(s, x):
        B, T, _ = x.shape
        xi, z = s.in_proj(x).chunk(2, -1)
        xi = F.silu(s.conv(xi.transpose(1, 2))[..., :T]).transpose(1, 2)
        dt, Bm, Cm = s.x_proj(xi).split([s.dtr, s.N, s.N], -1)
        with nocast():
            dt = F.softplus(s.dt_proj(dt.float())); A = -torch.exp(s.A_log); xf = xi.float()
            a = torch.exp(dt[..., None] * A)
            b = dt[..., None] * Bm.float()[:, :, None, :] * xf[..., None]
            h = lin_scan(a, b)
            y = (h * Cm.float()[:, :, None, :]).sum(-1) + s.D * xf
        return s.out_proj(y * F.silu(z.float()))

class SwiGLU(nn.Module):
    def __init__(s, d, h=None):
        super().__init__(); h = h or int(8 * d / 3) // 32 * 32
        s.w13 = nn.Linear(d, 2 * h, bias=False); s.w2 = nn.Linear(h, d, bias=False)
    def forward(s, x): a, b = s.w13(x).chunk(2, -1); return s.w2(F.silu(a) * b)

def build(d):
    H = max(d // 32, 1); kvh = H // 2 if H % 2 == 0 else H
    mix = lambda i: Attn(d, H, kvh, rope=True) if i % 4 == 3 else Mamba(d)
    return LM(d, [Block(d, mix(i), SwiGLU(d)) for i in range(8)], ckpt=True)

In [ ]:
# ---- parameter fitting (width searched so total params ~ 10M), eval and training loop
NAMES = {NAME: 'Hybrid Mamba+Attn'}; MICRO = {NAME: 8}; LR = {NAME: 0.002}
make = lambda name, d: build(d)
def aux_loss(model): return 0.0

nparams = lambda m: sum(p.numel() for p in m.parameters())

def evaluate(model, val, T, bs, nb):
    model.eval(); g = torch.Generator().manual_seed(0); ls = []
    for _ in range(nb):
        x, y = get_batch(val, bs, T, g)
        with torch.autocast("cuda", dtype=torch.float16): lg = model(x)
        ls.append(F.cross_entropy(lg.float().view(-1, V), y.reshape(-1)).item())
    model.train(); return float(np.mean(ls))

def fit(name, target=10e6):
    best = None
    for d in range(64, 1537, 32):
        n = nparams(make(name, d))
        if best is None or abs(n - target) < abs(best[1] - target): best = (d, n)
        if n > target: break
    return make(name, best[0]), best[0]

def train(name, a):
    os.makedirs("results", exist_ok=True); os.makedirs("ckpt", exist_ok=True)
    if os.path.exists(f"results/{name}.json"): print(f"[{name}] done already, skipping"); return
    dev = "cuda"; torch.manual_seed(0)
    tr = torch.from_numpy(np.load("data/train.npy").astype(np.int16)).to(dev)
    va = torch.from_numpy(np.load("data/val.npy").astype(np.int16)).to(dev)
    model, d = fit(name, a.target); model = model.to(dev); n = nparams(model)
    print(f"[{name}] d={d} params={n/1e6:.2f}M")
    micro = MICRO.get(name, a.bs); accum = a.bs // micro; lr0 = LR.get(name, a.lr)
    dec = [p for k, p in model.named_parameters() if p.ndim >= 2 and k.split(".")[-1] in ("weight", "w1", "w2", "w3")]
    nod = [p for k, p in model.named_parameters() if not (p.ndim >= 2 and k.split(".")[-1] in ("weight", "w1", "w2", "w3"))]
    opt = torch.optim.AdamW([{"params": dec, "weight_decay": 0.1}, {"params": nod, "weight_decay": 0.0}], lr=lr0, betas=(0.9, 0.95))
    scaler = torch.amp.GradScaler("cuda"); hist = []; eval_time = 0.0; diverged = False
    torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize(); t0 = time.time()
    for step in range(a.steps):
        lr = lr0 * (step + 1) / 100 if step < 100 else lr0 * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * (step - 100) / max(1, a.steps - 100))))
        for gp in opt.param_groups: gp["lr"] = lr
        tot = 0.0
        for _ in range(accum):
            x, y = get_batch(tr, micro, SEQ)
            with torch.autocast("cuda", dtype=torch.float16):
                lg = model(x)
                loss = F.cross_entropy(lg.float().view(-1, V), y.reshape(-1))
                total = loss + 0.01 * aux_loss(model)
            scaler.scale(total / accum).backward(); tot += loss.item() / accum
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        if not math.isfinite(tot): print(f"[{name}] diverged at step {step}"); diverged = True; break
        if step % 100 == 0 or step == a.steps - 1:
            ev = None
            if step % 500 == 0 or step == a.steps - 1:
                torch.cuda.synchronize(); te = time.time(); ev = evaluate(model, va, SEQ, 32, 20); eval_time += time.time() - te
            hist.append({"step": step, "train": tot, "val": ev})
            print(f"[{name}] step {step:5d} train {tot:.3f}" + (f" val {ev:.3f}" if ev else "") + f" | {time.time()-t0:.0f}s", flush=True)
    torch.cuda.synchronize(); ttrain = time.time() - t0 - eval_time
    res = {"name": name, "label": NAMES[name], "d_model": d, "params": n, "steps": step + 1, "diverged": diverged,
           "train_time_s": ttrain, "tok_per_s": (step + 1) * a.bs * SEQ / ttrain,
           "peak_mem_gb": torch.cuda.max_memory_allocated() / 1e9, "history": hist}
    if name == "moe":
        ex = sum(p.numel() for k, p in model.named_parameters() if k.split(".")[-1] in ("w1", "w2", "w3") and p.ndim == 3)
        res["active_params"] = n - ex * 6 // 8
    if not diverged:
        res["val_loss"] = evaluate(model, va, SEQ, 32, 60); res["val_ppl"] = math.exp(res["val_loss"])
        res["val_bits_per_token"] = res["val_loss"] / math.log(2); res["len_eval"] = {}
        for T in (256, 512, 1024):  # trained at 256: length extrapolation
            try: res["len_eval"][T] = evaluate(model, va, T, 32 if T <= 256 else 8, 20)
            except Exception as e: res["len_eval"][T] = None; print(f"[{name}] len {T} failed: {type(e).__name__}")
        torch.save(model.state_dict(), f"ckpt/{name}.pt")
    json.dump(res, open(f"results/{name}.json", "w"), indent=1)
    print(f"[{name}] FINAL val_loss={res.get('val_loss')} ppl={res.get('val_ppl')} tok/s={res['tok_per_s']:.0f} mem={res['peak_mem_gb']:.1f}GB")
    del model, opt; torch.cuda.empty_cache()

class A: pass
a = A(); a.steps = STEPS; a.bs = 32; a.lr = 2e-3; a.target = 10e6
m, d = fit(NAME); print(f'width d={d}, params={nparams(m)/1e6:.2f}M'); del m

In [ ]:
train(NAME, a)

### Training curve and metrics

In [ ]:
import matplotlib.pyplot as plt
h = [r for r in res["history"]]
plt.figure(figsize=(7, 4)); plt.plot([r["step"] for r in h], [r["train"] for r in h], label="train")
v = [r for r in h if r["val"]]; plt.plot([r["step"] for r in v], [r["val"] for r in v], "o-", label="val")
plt.xlabel("step"); plt.ylabel("loss (nats/token)"); plt.title(f"{res['label']} - {res['params']/1e6:.2f}M params"); plt.legend(); plt.grid(alpha=.3)
plt.savefig(f"results/{NAME}_curve.png", dpi=150, bbox_inches="tight"); plt.show()
print({k: res[k] for k in ["params", "val_loss", "val_ppl", "val_bits_per_token", "tok_per_s", "peak_mem_gb", "len_eval"]})

### Generate text from the trained 10M model

In [ ]:
from tokenizers import Tokenizer
tok = Tokenizer.from_file("data/tok.json")
res = json.load(open(f"results/{NAME}.json"))
model = build(res["d_model"]); model.load_state_dict(torch.load(f"ckpt/{NAME}.pt")); model.cuda().eval()

@torch.no_grad()
def generate(prompt, n=150, temp=0.8, top_k=40):
    x = torch.tensor([tok.encode(prompt).ids]).cuda()
    for _ in range(n):
        with torch.autocast("cuda", dtype=torch.float16):
            lg = model(x[:, -SEQ:])[0, -1].float() / temp
        lg[lg < lg.topk(top_k).values[-1]] = -float("inf")
        x = torch.cat([x, torch.multinomial(F.softmax(lg, -1), 1)[None]], 1)
    return tok.decode(x[0].tolist())

for p in ["Once upon a time", "Tom and his dog", "The little girl was sad because"]:
    print(generate(p), "\n" + "-" * 60)

### Export for the portfolio

In [ ]:
# Package the trained 10M model for the portfolio (weights + results + curve)
import shutil
os.makedirs("portfolio", exist_ok=True)
for f in [f"ckpt/{NAME}.pt", f"results/{NAME}.json", f"results/{NAME}_curve.png"]: shutil.copy(f, "portfolio/")
shutil.make_archive(f"{NAME}_10m", "zip", "portfolio"); print(f"saved {NAME}_10m.zip (download from the file browser)")